<a href="https://colab.research.google.com/github/KamoTaueatsoala/ML-Projects/blob/main/Irrigation_schedule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
import os

# Configuration Constants for Parsley (Herb)
MAX_MOISTURE = 65          # Parsley doesn't like soggy soil
MAX_HUMIDITY = 100
OPTIMAL_TEMP = 18          # Parsley prefers cooler temperatures
MIN_TEMP = 5
MAX_TEMP = 30
MAX_WATER = 2.0            # Less water than tomatoes
MODEL_PATH = "parsley_model.tflite"
HEADER_PATH = "model.h"

def generate_improved_training_data(samples=50000):
    np.random.seed(42)
    moisture = np.clip(0.7 * np.random.normal(42, 6, samples) + 0.3 * np.random.beta(2, 5, samples) * MAX_MOISTURE, 15, MAX_MOISTURE)
    humidity = np.clip(np.random.normal(60, 10, samples), 30, MAX_HUMIDITY)
    temp = np.clip(OPTIMAL_TEMP + np.random.normal(0, 2.0, samples) + 2 * np.sin(np.linspace(0, 10 * np.pi, samples)), MIN_TEMP, MAX_TEMP)
    hour = np.random.randint(0, 24, samples)
    day_of_year = np.random.randint(0, 365, samples)

    water = np.zeros(samples)
    for i in range(samples):
        if moisture[i] >= MAX_MOISTURE * 0.95 or temp[i] < MIN_TEMP + 2 or temp[i] > MAX_TEMP - 2:
            water[i] = 0
        else:
            base = np.sqrt((MAX_MOISTURE - moisture[i]) / MAX_MOISTURE)
            temp_ratio = abs(temp[i] - OPTIMAL_TEMP) / (MAX_TEMP - OPTIMAL_TEMP)
            temp_factor = np.exp(-2.2 * temp_ratio ** 1.5)
            humidity_factor = 1.1 - (humidity[i] / MAX_HUMIDITY) ** 0.4

            if 5 <= hour[i] <= 19:
                water[i] = base * temp_factor * humidity_factor * MAX_WATER * (0.80 + 0.05 * np.sin(hour[i] / 24 * 2 * np.pi))
            else:
                water[i] = base * temp_factor * humidity_factor * MAX_WATER * (0.30 + 0.05 * np.cos(hour[i] / 24 * 2 * np.pi))

            water[i] = np.clip(water[i] * (1 + np.random.normal(0, 0.02)), 0, MAX_WATER)

    features = np.column_stack((
        moisture / MAX_MOISTURE,
        (temp - OPTIMAL_TEMP) / (MAX_TEMP - OPTIMAL_TEMP),
        humidity / MAX_HUMIDITY,
        np.sin(2 * np.pi * hour / 24),
        np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * day_of_year / 365),
        np.cos(2 * np.pi * day_of_year / 365),
        (moisture * humidity) / (MAX_MOISTURE * MAX_HUMIDITY),
        np.sqrt(moisture * humidity) / np.sqrt(MAX_MOISTURE * MAX_HUMIDITY),
        (temp - OPTIMAL_TEMP) ** 3 / (MAX_TEMP - OPTIMAL_TEMP) ** 3,
        np.log1p(moisture) / np.log(MAX_MOISTURE),
        hour / 24,
        (humidity * (temp - OPTIMAL_TEMP)) / (MAX_HUMIDITY * (MAX_TEMP - OPTIMAL_TEMP))
    )).astype(np.float32)

    return features, water.astype(np.float32)

def create_quantizable_model(input_shape):
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.BatchNormalization()(inputs)
    x = keras.layers.Dense(96, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.15)(x)
    x = keras.layers.Dense(64, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    output = keras.layers.Dense(1, activation='linear')(x)

    model = keras.Model(inputs=inputs, outputs=output)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='huber', metrics=['mae'])
    return model

def train_model(X_train, y_train):
    model = create_quantizable_model(input_shape=(13,))
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ]
    model.fit(X_train, y_train, epochs=200, batch_size=256, validation_split=0.1, callbacks=callbacks, verbose=1)
    return model

def evaluate_model(model, X_test, y_test, label="Model"):
    preds = model.predict(X_test).flatten()
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{label} - MAE: {mae:.4f}L, R²: {r2:.4f}")
    return mae, r2

def convert_to_tflite(model, X_train):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    def representative_dataset_gen():
        for i in range(300):
            yield [X_train[i:i+1].astype(np.float32)]

    converter.representative_dataset = representative_dataset_gen
    tflite_model = converter.convert()
    with open(MODEL_PATH, 'wb') as f:
        f.write(tflite_model)
    print(f"\nModel saved as {MODEL_PATH} ({len(tflite_model) / 1024:.1f} KB)")
    return tflite_model

def export_as_c_header(tflite_model):
    with open(HEADER_PATH, "w") as f:
        f.write("#ifndef MODEL_H\n#define MODEL_H\n\n")
        f.write("const unsigned char model[] = {\n  ")
        bytes_per_line = 12
        for i in range(0, len(tflite_model), bytes_per_line):
            line = ", ".join(f"0x{b:02x}" for b in tflite_model[i:i+bytes_per_line])
            f.write(line + (",\n  " if i + bytes_per_line < len(tflite_model) else "\n"))
        f.write("};\n\n")
        f.write(f"const unsigned int model_len = {len(tflite_model)};\n\n#endif")
    print(f"Exported model to {HEADER_PATH}")

def train_and_convert():
    X, y = generate_improved_training_data()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

    print("Training base model...")
    base_model = train_model(X_train, y_train)
    evaluate_model(base_model, X_test, y_test, "Base Model")

    try:
        import tensorflow_model_optimization as tfmot
        print("\nQuantization-aware training enabled...")
        q_model = tfmot.quantization.keras.quantize_model(base_model)
        q_model.compile(optimizer='adam', loss='huber', metrics=['mae'])
        q_model.fit(X_train, y_train, epochs=20, batch_size=512, validation_split=0.1, verbose=1)
        evaluate_model(q_model, X_test, y_test, "Quantized Model")
        final_model = q_model
    except (ImportError, ValueError) as e:
        print(f"QAT not available: {e}. Falling back to float model.")
        final_model = base_model

    tflite_model = convert_to_tflite(final_model, X_train)
    export_as_c_header(tflite_model)

if __name__ == "__main__":
    try:
        import tensorflow_model_optimization
    except ImportError:
        print("Installing tensorflow-model-optimization...")
        import subprocess
        subprocess.check_call(["pip", "install", "tensorflow-model-optimization"])

    print("Starting training with humidity-enhanced features for parsley...")
    train_and_convert()


Starting training with humidity-enhanced features for parsley...
Training base model...
Epoch 1/200
159/159 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.1535 - mae: 0.4109 - val_loss: 0.0095 - val_mae: 0.0952 - learning_rate: 5.0000e-04
Epoch 2/200
159/159 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0254 - mae: 0.1659 - val_loss: 0.0078 - val_mae: 0.0830 - learning_rate: 5.0000e-04
Epoch 3/200
159/159 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0150 - mae: 0.1240 - val_loss: 0.0060 - val_mae: 0.0676 - learning_rate: 5.0000e-04
Epoch 4/200
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0107 - mae: 0.1007 - val_loss: 0.0053 - val_mae: 0.0605 - learning_rate: 5.0000e-04
Epoch 5/200
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0083 - mae: 0.0854 - val_loss: 0.0046 - val_mae: 0.0533 - learning_rate: 5.0000e-04
Epoch 6/200
159/159 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0070 - mae: 0.0758 - val_loss: 0.0042 - val_mae: 0.0484 - learning_rate: 5.0000e-04
Epoch 7/200
159/159 ━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
